# Agent Observability

## Scenario: explain a Northstar incident trajectory

A checkout investigation routes to a bounded agent, invokes model triage and read-only tools, then creates an approval-required proposal. We instrument every decision so an operator can answer why it happened—not merely read the final response.

**Safety boundary:** traces are privacy-aware. Record versions, IDs, classes, and metrics by default; redact/hash sensitive prompt, tool, and customer content.

![Agent observability trace](../../../assets/agent-observability-trace.svg)

Use OpenTelemetry-style root traces and child spans for route, model/context, tools, state/policy, output, evaluator, and errors. Correlate with request/run/session and tenant-safe identifiers.

In [1]:
from pathlib import Path
import sys
TOPIC=Path.cwd()
if not (TOPIC/'lab.py').exists(): TOPIC=Path.cwd()/'curriculum'/'enterprise-agent'/'13-agent-observability'
sys.path.insert(0,str(TOPIC))
from lab import incident_trace,summarize
spans=incident_trace()
for span in spans: print(span.name,span.kind,span.latency_ms,'ms',span.tokens,'tokens',span.cost_cents,'cents',span.attrs)
summary=summarize(spans); print(summary)
assert summary['trajectory']==['route','model.triage','tool.metrics','tool.logs','policy.proposal']

route internal 12 ms 0 tokens 0 cents {'route': 'single-agent', 'tenant': 'acme'}
model.triage llm 540 ms 620 tokens 0.11 cents {'prompt_version': 'v3', 'context_items': 3}
tool.metrics tool 180 ms 0 tokens 0 cents {'tool': 'get_service_metrics'}
tool.logs tool 420 ms 0 tokens 0 cents {'tool': 'query_logs'}
policy.proposal internal 15 ms 0 tokens 0 cents {'decision': 'approval-required'}
{'latency_ms': 1167, 'tokens': 620, 'cost_cents': 0.11, 'errors': [], 'trajectory': ['route', 'model.triage', 'tool.metrics', 'tool.logs', 'policy.proposal']}


## Debugging, replay, monitoring, and production dashboards

Inspect route reason, prompt/context version and item counts, tool schema/result class, state/checkpoint, policy/approval, retry/error, tokens, cost, and span latency. Classify failure as model/provider, tool/transient, schema, evidence/retrieval, policy/auth, budget/timeout, state/recovery, or user correction. Replay deterministic fixtures; never replay side effects without fresh authorization, idempotency, and approval.

Dashboards: success/evidence/policy rates; p50/p95/p99 route/model/tool/queue/total latency; tokens/cost per successful safe task; fallback/retry/error class; approval delay; trace coverage; model/prompt/tool drift.

**Exercises:** add a timeout span, correlate an evaluation failure with a trace, redact an attribute, design an OpenTelemetry exporter schema, and set an alert for policy blocks or tool latency.

References: [OpenTelemetry](https://opentelemetry.io/docs/), [OpenAI Agents tracing](https://openai.github.io/openai-agents-python/tracing/), [Phoenix](https://docs.arize.com/phoenix), [LangSmith](https://docs.smith.langchain.com/observability).